In [1]:
import pandas as pd

In [3]:
# Load Dataset
df = pd.read_csv("financial_market_dirty.csv")

In [4]:
# Inspect Dataset
print(df.head())
print(df.info())
print(df.shape)

     Trade_ID          Date Ticker           Company_Name         Sector  \
0  TRD-134945    2024-07-26   AAPL             Apple Inc.     Technology   
1  TRD-123987  Jun 16, 2023   TSLA             Tesla Inc.     Automotive   
2  TRD-102884    2021-04-22   MSFT  Microsoft Corporation     Technology   
3  TRD-122847  May 05, 2023   NVDA     NVIDIA Corporation     Technology   
4  TRD-125704    2023-08-21    DIS        Walt Disney Co.  Communication   

  Exchange Currency    Open    High     Low  ... Trade_Price  Quantity  \
0   NASDAQ      USD  124.47  125.40  123.73  ...      123.92     422.0   
1     NYSE      CAD  405.81  405.97  398.46  ...      405.73       NaN   
2   NASDAQ      USD  260.47  262.80  254.90  ...      262.43     369.0   
3   NASDAQ      USD  698.21  699.16  696.77  ...      699.10      42.0   
4     NYSE      GBP  104.33  106.19  104.17  ...         NaN     136.0   

  Trade_Value    MA_7   MA_30    RSI  Trader_ID     Broker  Commission  \
0    52294.24  124.81   

In [5]:
# Standardize Column Names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [6]:
#  Remove Duplicate Rows

df.drop_duplicates(inplace=True)

In [11]:
# Remove Extra Whitespace

for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) or df[col].dtype == "object":
        df[col] = df[col].astype("string").str.strip()

In [13]:
# Replace Invalid Values

invalid_values = ["N/A", "#VALUE!", "", "null", "None", "nan"]

df.replace(invalid_values, pd.NA, inplace=True)

,trade_id,date,ticker,company_name,sector,exchange,currency,open,high,low,...,trade_price,quantity,trade_value,ma_7,ma_30,rsi,trader_id,broker,commission,net_value
0,TRD-134945,2024-07-26,AAPL,Apple Inc.,Technology,NASDAQ,USD,124.47,125.40,123.73,...,123.92,422.0,52294.24,124.81,NaN,49.04,TRD1057,Fidelity,52.29,52241.95
1,TRD-123987,"Jun 16, 2023",TSLA,Tesla Inc.,Automotive,NYSE,CAD,405.81,405.97,398.46,...,405.73,NaN,198807.7,403.33,400.04,69.58,TRD1032,Robinhood,198.81,198608.89
2,TRD-102884,2021-04-22,MSFT,Microsoft Corporation,Technology,NASDAQ,USD,260.47,262.80,254.90,...,262.43,369.0,96836.67,260.10,256.93,47.57,TRD1031,Fidelity,96.84,96739.83
3,TRD-122847,"May 05, 2023",NVDA,NVIDIA Corporation,Technology,NASDAQ,USD,698.21,699.16,696.77,...,699.10,42.0,29362.2,700.76,694.71,37.01,TRD1046,JP Morgan,29.36,29332.84
4,TRD-125704,2023-08-21,DIS,Walt Disney Co.,Communication,NYSE,GBP,104.33,106.19,104.17,...,NaN,136.0,14207.92,105.85,110.31,42.11,TRD1008,<NA>,14.21,14193.71
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10345,TRD-100053,2021-01-05,WMT,Walmart Inc.,Consumer Defensive,TSX,<NA>,161.81,162.35,158.45,...,160.16,234.0,37477.44,159.70,151.53,61.35,TRD1075,Schwab,37.48,37439.96
10346,TRD-106169,24-08-2021,JNJ,Johnson & Johnson,Healthcare,LSE,USD,201.13,202.32,191.17,...,195.41,259.0,50611.19,190.35,188.94,54.96,TRD1037,Fidelity,50.61,50560.58
10347,TRD-114218,2022-06-20,MSFT,Microsoft Corporation,Technology,NASDAQ,USD,517.57,524.56,515.24,...,NaN,360.0,187970.4,528.94,523.38,NaN,TRD1030,Fidelity,187.97,187782.43
10348,TRD-123739,2023-06-07,NVDA,NVIDIA Corporation,Technology,TSX,USD,760.68,765.93,748.52,...,763.36,154.0,117557.44,745.15,759.26,64.10,TRD1033,Morgan Stanley,117.56,117439.88


In [14]:
# Convert Date Column

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

In [15]:
#  Remove Currency Symbols

price_columns = ["open", "high", "low", "close"]

for col in price_columns:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[$,A-Za-z]", "", regex=True)
        )

In [16]:
# Convert Numeric Columns

numeric_cols = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "rsi"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [17]:
# Standardize Stock Tickers

if "ticker" in df.columns:
    df["ticker"] = df["ticker"].str.upper()

In [18]:
# Standardize Trade Type

if "trade_type" in df.columns:
    mapping = {
        "BUY": "BUY",
        "B": "BUY",
        "1": "BUY",
        "SELL": "SELL",
        "S": "SELL",
        "0": "SELL"
    }

    df["trade_type"] = (
        df["trade_type"]
        .str.upper()
        .map(mapping)
    )


In [21]:
# Fix Impossible Prices

df.loc[df[col] <= 0, col] = pd.NA

In [23]:
#  Validate RSI (0-100)

if "rsi" in df.columns:
    df.loc[(df["rsi"] < 0) | (df["rsi"] > 100), "rsi"] = pd.NA

In [24]:
# Validate OHLC Logic

if {"high", "low"}.issubset(df.columns):
    df = df[df["high"] >= df["low"]]


In [25]:
# Remove Special Characters

if "trade_id" in df.columns:
    df["trade_id"] = df["trade_id"].str.replace(
        r"[^A-Za-z0-9-]",
        "",
        regex=True
    )

In [27]:
#  Handle Missing Values

df.ffill(inplace=True)

,trade_id,date,ticker,company_name,sector,exchange,currency,open,high,low,...,trade_price,quantity,trade_value,ma_7,ma_30,rsi,trader_id,broker,commission,net_value
0,TRD-134945,2024-07-26,AAPL,Apple Inc.,Technology,NASDAQ,USD,124.47,125.40,123.73,...,123.92,422.0,52294.24,124.81,NaN,49.04,TRD1057,Fidelity,52.29,52241.95
1,TRD-123987,2024-07-26,TSLA,Tesla Inc.,Automotive,NYSE,CAD,405.81,405.97,398.46,...,405.73,422.0,198807.7,403.33,400.04,69.58,TRD1032,Robinhood,198.81,198608.89
2,TRD-102884,2021-04-22,MSFT,Microsoft Corporation,Technology,NASDAQ,USD,260.47,262.80,254.90,...,262.43,369.0,96836.67,260.10,256.93,47.57,TRD1031,Fidelity,96.84,96739.83
3,TRD-122847,2021-04-22,NVDA,NVIDIA Corporation,Technology,NASDAQ,USD,698.21,699.16,696.77,...,699.10,42.0,29362.2,700.76,694.71,37.01,TRD1046,JP Morgan,29.36,29332.84
4,TRD-125704,2023-08-21,DIS,Walt Disney Co.,Communication,NYSE,GBP,104.33,106.19,104.17,...,699.10,136.0,14207.92,105.85,110.31,42.11,TRD1008,JP Morgan,14.21,14193.71
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10345,TRD-100053,2021-01-05,WMT,Walmart Inc.,Consumer Defensive,TSX,GBP,161.81,162.35,158.45,...,160.16,234.0,37477.44,159.70,151.53,61.35,TRD1075,Schwab,37.48,37439.96
10346,TRD-106169,2021-01-05,JNJ,Johnson & Johnson,Healthcare,LSE,USD,201.13,202.32,191.17,...,195.41,259.0,50611.19,190.35,188.94,54.96,TRD1037,Fidelity,50.61,50560.58
10347,TRD-114218,2022-06-20,MSFT,Microsoft Corporation,Technology,NASDAQ,USD,517.57,524.56,515.24,...,195.41,360.0,187970.4,528.94,523.38,54.96,TRD1030,Fidelity,187.97,187782.43
10348,TRD-123739,2023-06-07,NVDA,NVIDIA Corporation,Technology,TSX,USD,760.68,765.93,748.52,...,763.36,154.0,117557.44,745.15,759.26,64.10,TRD1033,Morgan Stanley,117.56,117439.88


In [28]:
# Final Check

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nData Types")
print(df.dtypes)


Missing Values
trade_id        0
date            0
ticker          0
company_name    0
sector          0
exchange        0
currency        0
open            0
high            0
low             0
close           0
volume          0
trade_type      0
trade_price     0
quantity        0
trade_value     0
ma_7            0
ma_30           1
rsi             0
trader_id       0
broker          0
commission      0
net_value       0
dtype: int64

Duplicate Rows
15

Data Types
trade_id                string
date            datetime64[us]
ticker                  string
company_name            string
sector                  string
exchange                string
currency                string
open                   float64
high                   float64
low                    float64
close                  float64
volume                 float64
trade_type                 str
trade_price            float64
quantity               float64
trade_value             string
ma_7                   float64

In [29]:
# Missing Value
df["ma_30"] = df["ma_30"].ffill().bfill()

# or
df["ma_30"] = df["ma_30"].fillna(df["ma_30"].median())

In [30]:
# 15 duplicate rows
print("Before:", df.shape)

df = df.drop_duplicates()

print("After:", df.shape)

Before: (10217, 23)
After: (10202, 23)


In [31]:
# Check duplicates
print(df.duplicated().sum())

0


In [32]:
# Check datatype
print(df["trade_value"].dtype)

string


In [33]:
# Convert right datatype
df["trade_value"] = (
    df["trade_value"]
    .astype("string")
    .str.replace(r"[$,]", "", regex=True)
)

df["trade_value"] = pd.to_numeric(df["trade_value"], errors="coerce")

In [34]:
# Check datatype
print(df["trade_value"].dtype)

Float64


In [36]:
df["trade_type"] = df["trade_type"].astype("string")

In [37]:
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.dtypes)

trade_id          0
date              0
ticker            0
company_name      0
sector            0
exchange          0
currency          0
open              0
high              0
low               0
close             0
volume            0
trade_type        0
trade_price       0
quantity          0
trade_value     197
ma_7              0
ma_30             0
rsi               0
trader_id         0
broker            0
commission        0
net_value         0
dtype: int64
3
trade_id                string
date            datetime64[us]
ticker                  string
company_name            string
sector                  string
exchange                string
currency                string
open                   float64
high                   float64
low                    float64
close                  float64
volume                 float64
trade_type              string
trade_price            float64
quantity               float64
trade_value            Float64
ma_7                   float6

In [38]:
# Fill missing trade_value using trade_price × quantity
mask = df["trade_value"].isna()

In [39]:
df.loc[mask, "trade_value"] = (
    df.loc[mask, "trade_price"] * df.loc[mask, "quantity"]
)

In [40]:
df["trade_value"] = (
    df["trade_value"]
    .astype("string")
    .str.replace(r"[$,]", "", regex=True)
)

df["trade_value"] = pd.to_numeric(df["trade_value"], errors="coerce")

In [41]:
mask = df["trade_value"].isna()

df.loc[mask, "trade_value"] = (
    df.loc[mask, "trade_price"] * df.loc[mask, "quantity"]
)

In [42]:
print(df["trade_value"].isna().sum())

0


In [43]:
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.dtypes)

trade_id        0
date            0
ticker          0
company_name    0
sector          0
exchange        0
currency        0
open            0
high            0
low             0
close           0
volume          0
trade_type      0
trade_price     0
quantity        0
trade_value     0
ma_7            0
ma_30           0
rsi             0
trader_id       0
broker          0
commission      0
net_value       0
dtype: int64
8
trade_id                string
date            datetime64[us]
ticker                  string
company_name            string
sector                  string
exchange                string
currency                string
open                   float64
high                   float64
low                    float64
close                  float64
volume                 float64
trade_type              string
trade_price            float64
quantity               float64
trade_value            Float64
ma_7                   float64
ma_30                  float64
rsi          

In [44]:
# Save Clean Dataset

df.to_csv("Financial_Market_Cleaned.csv", index=False)

print("\n✅ Data cleaning completed successfully!")


✅ Data cleaning completed successfully!
